# Deep Learning Framework Training Demo

This notebook provides a complete environment for:
1. Checking system information
2. Building the C++ framework and Python bindings
3. Running a mini-training (1-2 batches) for verification

## 1. System Information

In [ ]:
import os
import sys
import platform
import multiprocessing
import subprocess

print(f"Operating System: {platform.system()} {platform.release()}")
print(f"Python Version: {sys.version}")
print(f"CPU Cores: {multiprocessing.cpu_count()}")
print(f"Current Working Directory: {os.getcwd()}")

## 2. Install Dependencies & Build Framework
If you are on Colab, you need to install CMake and OpenCV development headers.

In [ ]:
# Verify and load bindings
sys.path.append(os.path.abspath('build/bindings'))
try:
    import deeplearn as dl
    import numpy as np
    import cv2
    from python.dataloader import DataLoader
    from python.model import SimpleCNN
    print('Success: Framework and local modules loaded.')
except ImportError as e:
    print(f'Import Error: {e}')
    print('Common fixes:')
    print("1. Ensure 'build/bindings' contains the .so file.")
    print("2. Run 'pip3 list | grep opencv install opencv-python'.")
    print("3. Verify the 'python' folder exists in the current directory.")

## 3. Verified Mini-Training (2 Batches)
We run just 2 batches to verify everything works correctly while saving time and credits.

In [ ]:
def run_mini_train(dataset_name):
    if not os.path.exists(dataset_name):
        print(f"Error: Dataset directory '{dataset_name}' not found.")
        return
        
    print(f"
--- Verification on {dataset_name} (2 Batches) ---")
    data_dir = dataset_name
    try:
        loader = DataLoader(data_dir, batch_size=32, shuffle=True)
        model = SimpleCNN(num_classes=len(loader.classes))
        optimizer = dl.SGD(model.parameters(), lr=0.01)
        criterion = dl.CrossEntropyLoss()
        
        import time
        start = time.time()
        
        batch_count = 0
        for images, labels, load_time in loader:
            batch_count += 1
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            print(f"  Batch {batch_count}: Loss = {loss.to_numpy()[0]:.4f}")
            
            if batch_count >= 2:
                print(f"  Reached 2 batches. Verification successful!")
                break
                
        print(f"Time taken for 2 batches: {time.time() - start:.2f} seconds")
    except Exception as e:
        print(f"Training error: {e}")

run_mini_train('data_1')
run_mini_train('data_2')

## 4. Run Full Training (If desired)
To run the full training as per the original script:
`export PYTHONPATH=$(pwd)/build/bindings && python3 python/train.py --dataset data_1 --epochs 1`